The Washington State Department of Licensing provides detailed records of all registered electric vehicles (EVs), including attributes such as County, City, Postal Code, Model Year, Make, Model, Electric Vehicle Type (BEV/PHEV), Clean Alternative Fuel Vehicle Eligibility, Electric Range, Base MSRP, Legislative District, Electric Utility, and Census Tract.

Goal:
Develop a predictive model that estimates the electric driving range of a newly registered EV given its make, model year, type (BEV or PHEV), base MSRP, and geographic features (county, utility provider, census tract).

In [ ]:
# -----------------------------
# 1. Importing Libraries
# -----------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

In [ ]:
# -----------------------------
# 2. Loading Dataset
# -----------------------------
dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Electric_Vehicle_Population_Data.csv')

In [ ]:
dataset.columns

In [ ]:
dataset.describe

In [ ]:
dataset.head(5)


In [ ]:
# -----------------------------
# 3. Data Preprocessing
# -----------------------------
# Droping unnecessary identifiers
dataset = dataset.drop(columns=['VIN (1-10)', 'DOL Vehicle ID', 'Vehicle Location'], errors='ignore')

# Handle missing values
dataset['Base MSRP'] = dataset['Base MSRP'].fillna(dataset['Base MSRP'].median())
dataset['Electric Range'] = dataset['Electric Range'].fillna(dataset['Electric Range'].median())
dataset['Electric Utility'] = dataset['Electric Utility'].fillna("Unknown")
dataset['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] = dataset['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].fillna("Unknown")

# Identify columns with remaining missing values
cols_with_nan = dataset.columns[dataset.isnull().any()].tolist()

# Impute missing values in identified columns with the median
for col in cols_with_nan:
    if dataset[col].dtype in ['int64', 'float64']:
        dataset[col] = dataset[col].fillna(dataset[col].median())
    else:
        dataset[col] = dataset[col].fillna("Unknown")


# Feature engineering: Vehicle Age
dataset['Vehicle_Age'] = 2025 - dataset['Model Year']

# Target variable
y = dataset['Electric Range']

# Features
X = dataset.drop(columns=['Electric Range'])

In [ ]:

# -----------------------------
# 4. Encoding Categorical Features
# -----------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])


In [ ]:

# -----------------------------
# 5. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:

# -----------------------------
# 6. Modeling
# -----------------------------

# Decision Tree
dt_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', DecisionTreeRegressor(random_state=42))])
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

# Random Forest
rf_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# Neural Network
mlp_model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('model', MLPRegressor(hidden_layer_sizes=(64, 32),
                                                   max_iter=300, random_state=42))])
mlp_model.fit(X_train, y_train)
mlp_preds = mlp_model.predict(X_test)


In [ ]:

# -----------------------------
# 7. Evaluation
# -----------------------------
def evaluate(y_true, y_pred, model_name):
    print(f"\n?? {model_name} Results")
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("R?:", r2_score(y_true, y_pred))

evaluate(y_test, dt_preds, "Decision Tree")
evaluate(y_test, rf_preds, "Random Forest")
evaluate(y_test, mlp_preds, "Neural Network")


In [ ]:

# -----------------------------
# 8. Visualization
# -----------------------------
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8,8))

# If you have the original feature column available for hue (optional)
if 'Electric Vehicle Type' in X_test.columns:
    sns.scatterplot(
        x=y_test,
        y=rf_preds,
        hue=X_test['Electric Vehicle Type'],
        palette='Set2',
        alpha=0.6,
        edgecolor=None
    )
    plt.legend(title="EV Type")
else:
    # fallback if that column isn't present
    plt.scatter(y_test, rf_preds, alpha=0.6, edgecolors='none')

# 45-degree reference line
line_start = min(y_test.min(), rf_preds.min())
line_end   = max(y_test.max(), rf_preds.max())
plt.plot([line_start, line_end], [line_start, line_end], 'r--', lw=2)

plt.xlabel("Actual Electric Range (miles)")
plt.ylabel("Predicted Electric Range (RF)")
plt.title("Random Forest Predictions vs Actual")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
errors = y_test - rf_preds
plt.figure(figsize=(7,4))
sns.histplot(errors, kde=True, bins=30, color='steelblue')
plt.axvline(0, color='red', linestyle='--')
plt.title("Prediction Error Distribution")
plt.xlabel("Actual ? Predicted Range (miles)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=X_test['Electric Vehicle Type'], y=(y_test - rf_preds))
plt.axhline(0, color='red', linestyle='--')
plt.title("Prediction Error by EV Type")
plt.xlabel("EV Type")
plt.ylabel("Error (Actual ? Predicted)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,6))
sns.residplot(x=rf_preds, y=y_test, lowess=True,
              scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.xlabel("Predicted Range")
plt.ylabel("Residual (Actual ? Predicted)")
plt.title("Residuals vs Predicted")
plt.axhline(0, color='black', lw=1)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 9. EV Range Clustering  (KMeans + Gaussian Mixture Model)
# ============================================================

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# We'll cluster only on a few meaningful numeric features
cluster_features = dataset[['Base MSRP', 'Vehicle_Age', 'Electric Range']].copy()

# --- Standardize ---
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

# ------------------- KMEANS -------------------
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(cluster_scaled)

dataset['KMeans_Cluster'] = kmeans_labels

# Optional: label clusters by average range
cluster_range_means = dataset.groupby('KMeans_Cluster')['Electric Range'].mean().sort_values()
range_order = {cluster: idx for idx, cluster in enumerate(cluster_range_means.index)}
dataset['Range_Segment'] = dataset['KMeans_Cluster'].map(range_order)

print("\nK-Means Range Means:")
print(cluster_range_means)

# ------------------- GMM -------------------
gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(cluster_scaled)
dataset['GMM_Cluster'] = gmm_labels

print("\nGMM Cluster Sample:")
print(dataset[['Base MSRP', 'Vehicle_Age', 'Electric Range', 'GMM_Cluster']].head())

# ------------------- Visualization -------------------
plt.figure(figsize=(7,6))
sns.scatterplot(
    x='Base MSRP',
    y='Electric Range',
    hue='KMeans_Cluster',
    palette='Set2',
    data=dataset,
    alpha=0.6
)
plt.title("EV Clusters by K-Means")
plt.xlabel("Base MSRP")
plt.ylabel("Electric Range")
plt.legend(title="Cluster")
plt.show()
